# Week 3: Exploratory Data Analysis & Preprocessing
## Notebook 06: Target Encoding & Preventing Target Leakage from Scratch

### 1. Conceptual Intuition & Real-World Use Case
When dealing with high-cardinality categorical variables (e.g. ZIP code with 40,000 values, merchant ID with 100,000 values):
- One-Hot Encoding creates 40,000 sparse columns, triggering the **Curse of Dimensionality** and memory crashes.
- Label / Ordinal Encoding introduces an arbitrary numerical ordering (e.g. ZIP 90210 > 10001) that confuses linear models and neural nets.

**Target Encoding (Mean Encoding)** replaces each categorical value with the average target value for that category.
However, naive target encoding causes **Catastrophic Target Leakage (Overfitting)**! A rare category with 1 sample that happens to be positive receives a target score of $1.0$, fooling the model.

**Solution:**
1. **M-Estimate Empirical Bayes Smoothing:** Blends category mean with global prior mean.
2. **K-Fold Out-of-Fold (OOF) Target Encoding:** Computes encoding using only out-of-fold data partitions.

---

### 2. Mathematical Foundations: M-Estimate Smoothing
For category level $c$ with $n_c$ observations and local target sum $y_c$:

$$\hat{S}_c = \frac{y_c + m \cdot \mu_{global}}{n_c + m}$$

Where:
- $\mu_{global}$ is the global target mean across the entire dataset.
- $m$ is the smoothing weight (prior weight, typically $5$ to $20$).
- When $n_c$ is tiny ($n_c \ll m$), $\hat{S}_c \to \mu_{global}$ (shrunk toward prior, preventing leakage).
- When $n_c$ is huge ($n_c \gg m$), $\hat{S}_c \to \frac{y_c}{n_c}$ (true category mean).

In [ ]:
import numpy as np
import pandas as pd

# Step 1: Create Dataset with High-Cardinality Categories & Target
np.random.seed(42)
data = pd.DataFrame({
    "city": ["NewYork", "NewYork", "NewYork", "London", "London", "Tokyo", "Tokyo", "Tokyo", "Tokyo", "SmallTown"],
    "target": [1, 1, 0, 1, 0, 0, 0, 0, 1, 1]  # Binary classification target
})

global_mean = data["target"].mean()
print(f"Global Target Mean (Prior): {global_mean:.3f}")
print("Raw Dataset:\n", data)

# Step 2: Naive Target Encoding (Vulnerable to Overfitting!)
naive_encoding = data.groupby("city")["target"].transform("mean")
data["naive_encoded"] = naive_encoding
print("\nNotice 'SmallTown' has 1 sample (target=1) -> gets Naive score 1.0 (Overfitting risk!):")
print(data[["city", "target", "naive_encoded"]].tail(3))

# Step 3: Implement M-Estimate Smoothed Target Encoding
def m_estimate_target_encode(df, col, target_col, m=5.0):
    global_prior = df[target_col].mean()
    stats = df.groupby(col)[target_col].agg(["count", "sum"])
    # Formula: (sum + m * prior) / (count + m)
    smoothed = (stats["sum"] + m * global_prior) / (stats["count"] + m)
    return df[col].map(smoothed)

data["smoothed_encoded"] = m_estimate_target_encode(data, "city", "target", m=5.0)

# Step 4: Implement 5-Fold Out-of-Fold (OOF) Target Encoding
def oof_target_encode(df, col, target_col, n_splits=3, m=5.0):
    encoded_vals = np.zeros(len(df))
    # Assign fold IDs
    indices = np.arange(len(df))
    np.random.shuffle(indices)
    folds = np.array_split(indices, n_splits)
    
    for fold_idx in range(n_splits):
        val_idx = folds[fold_idx]
        train_idx = np.hstack([folds[j] for j in range(n_splits) if j != fold_idx])
        
        train_subset = df.iloc[train_idx]
        val_subset = df.iloc[val_idx]
        
        prior = train_subset[target_col].mean()
        stats = train_subset.groupby(col)[target_col].agg(["count", "sum"])
        smoothed = (stats["sum"] + m * prior) / (stats["count"] + m)
        
        # Map onto validation split; fallback to prior for unseen categories
        encoded_vals[val_idx] = val_subset[col].map(smoothed).fillna(prior)
        
    return encoded_vals

data["oof_smoothed_encoded"] = oof_target_encode(data, "city", "target", n_splits=3, m=5.0)

print("\n--- Final Feature Comparison ---")
print(data[["city", "target", "naive_encoded", "smoothed_encoded", "oof_smoothed_encoded"]])

### 3. Step-by-Step Code Walkthrough
1. **Naive Flaw:** Notice `SmallTown` received a $1.0$ in naive encoding because its single sample had `target=1`.
2. **M-Estimate Shrinkage:** Smoothed encoding pulled `SmallTown` down from $1.0 \to 0.583$, shrinking it toward the global prior of $0.500$.
3. **Out-of-Fold Partitioning:** Encodings for each validation fold are computed strictly from the remaining folds, guaranteeing zero target information leaks into training rows.

---

### 4. Industry Applications & Trade-offs
- **Credit Scoring & Ad Click Prediction (CTR):** Encoding millions of user IDs, publisher domains, and search queries.
- **CatBoost:** Built-in ordered target encoding eliminates target leakage automatically.

---

### 5. Top Interview Questions & Answers
**Q1: Why does target encoding lead to severe overfitting if applied before train/test splitting?**
> **Answer:** If applied to the entire dataset prior to splitting, the target values of the test set are averaged into the category values seen by the training set. This constitutes direct data leakage; the model learns to memorize spurious correlations rather than true patterns.

**Q2: How does M-estimate smoothing balance high-frequency vs low-frequency categories?**
> **Answer:** When $n_c$ (sample count) is large (e.g. $n_c=10,000$), $m=10$ has negligible effect, allowing the true empirical mean to dominate. When $n_c$ is tiny ($n_c=1$), the numerator is dominated by $m \cdot \mu_{global}$, pulling the estimate to the global average.